# Banker's Algorithm — animated explainer (Colab)

An explainer-video style animation of the Banker's Algorithm (deadlock avoidance):
numbers fly and count up, every `Need ≤ Work` check is coloured resource by resource,
and every step has a narration caption.

**How to use:** `Runtime → Run all`, or run the cells from top to bottom.

1. **Setup** installs pygame and writes the program files
2. **Choose the data**: one of the 6 samples, or type your own
3. **Watch the video**: renders an MP4 and plays it here (you can download it)
4. **Step through**: go step by step with a slider, without rendering a video

## 1. Setup

In [ ]:
#@title Install pygame (run once)
!pip -q install pygame
import os
os.environ["SDL_VIDEODRIVER"] = "dummy"   # no screen in Colab: draw off-screen
os.environ["SDL_AUDIODRIVER"] = "dummy"
os.environ["PYGAME_HIDE_SUPPORT_PROMPT"] = "1"

The next cells write the program's source files (the same files as the `python/` folder in the repo).
You don't need to read them; just run them.

In [ ]:
%%writefile banker.py
"""Banker's Algorithm core (no drawing code). Mirrors core/banker.js.

state = {"available": [m], "max": [[n][m]], "alloc": [[n][m]]}
Need is always computed as Max - Allocation, never stored.
safety() records every step so the animation can replay it.
"""
import copy


def leq(a, b):
    return all(x <= y for x, y in zip(a, b))


def add(a, b):
    return [x + y for x, y in zip(a, b)]


def sub(a, b):
    return [x - y for x, y in zip(a, b)]


def is_nonneg_int(v):
    return isinstance(v, int) and not isinstance(v, bool) and v >= 0


def compute_need(state):
    return [[mx - al for mx, al in zip(mrow, arow)] for mrow, arow in zip(state["max"], state["alloc"])]


def validate_state(s):
    """Return a list of error messages ([] means the state is usable)."""
    errors = []
    n, m = len(s.get("max", [])), len(s.get("available", []))
    if n == 0 or m == 0:
        return ["need at least 1 process and 1 resource"]
    if len(s.get("alloc", [])) != n:
        errors.append("Allocation and Max have a different number of rows")
        return errors
    for j, v in enumerate(s["available"]):
        if not is_nonneg_int(v):
            errors.append(f"Available[{j}] must be a non-negative integer")
    for i in range(n):
        if len(s["max"][i]) != m or len(s["alloc"][i]) != m:
            errors.append(f"P{i}: column count does not match the number of resources")
            continue
        for j in range(m):
            mx, al = s["max"][i][j], s["alloc"][i][j]
            if not is_nonneg_int(mx) or not is_nonneg_int(al):
                errors.append(f"P{i} column {j}: Max and Allocation must be non-negative integers")
            elif mx < al:
                errors.append(f"P{i} column {j}: Max ({mx}) < Allocation ({al}) makes Need negative")
    return errors


def safety(state):
    """Safety algorithm. Scans P0..Pn-1 repeatedly until nobody else can finish.

    Returns {"safe", "sequence", "steps"}; each step is a dict with
    kind in init|check|finish|done plus work/finish/sequence snapshots.
    """
    n = len(state["max"])
    need = compute_need(state)
    work = list(state["available"])
    finish = [False] * n
    sequence = []
    steps = []

    def snap(**kw):
        kw.update(work=list(work), finish=list(finish), sequence=list(sequence))
        steps.append(kw)

    snap(kind="init")
    progress = True
    while len(sequence) < n and progress:
        progress = False
        for i in range(n):
            if finish[i]:
                continue
            ok = leq(need[i], work)
            snap(kind="check", pid=i, ok=ok, need=list(need[i]))
            if ok:
                before = work
                work = add(work, state["alloc"][i])
                finish[i] = True
                sequence.append(i)
                progress = True
                snap(kind="finish", pid=i, before=list(before))
    safe = len(sequence) == n
    snap(kind="done", safe=safe, stuck=[i for i in range(n) if not finish[i]])
    return {"safe": safe, "sequence": sequence, "steps": steps}


def request(state, pid, req):
    """Resource-request algorithm.

    Returns {"status": granted|wait|denied|invalid, "code", "trial", "safety", "new_state"}.
    """
    m = len(state["available"])
    if not isinstance(pid, int) or not 0 <= pid < len(state["max"]):
        return {"status": "invalid", "code": "BAD_PID", "new_state": state}
    if len(req) != m or not all(is_nonneg_int(v) for v in req):
        return {"status": "invalid", "code": "BAD_REQUEST", "new_state": state}
    need = compute_need(state)[pid]
    if not leq(req, need):
        return {"status": "denied", "code": "EXCEEDS_NEED", "new_state": state}
    if not leq(req, state["available"]):
        return {"status": "wait", "code": "NOT_AVAILABLE", "new_state": state}
    trial = copy.deepcopy(state)
    trial["available"] = sub(trial["available"], req)
    trial["alloc"][pid] = add(trial["alloc"][pid], req)
    s = safety(trial)
    if s["safe"]:
        return {"status": "granted", "code": "GRANTED", "trial": trial, "safety": s, "new_state": trial}
    return {"status": "denied", "code": "UNSAFE", "trial": trial, "safety": s, "new_state": state}

In [ ]:
%%writefile samples.py
"""Sample scenarios with hand-checked answers (same data as samples/samples.js)."""

_BASE = {
    "available": [3, 3, 2],
    "max": [[7, 5, 3], [3, 2, 2], [9, 0, 2], [2, 2, 2], [4, 3, 3]],
    "alloc": [[0, 1, 0], [2, 0, 0], [3, 0, 2], [2, 1, 1], [0, 0, 2]],
}
# state after P1's request (1,0,2) was granted
_AFTER_P1 = {
    "available": [2, 3, 0],
    "max": _BASE["max"],
    "alloc": [[0, 1, 0], [3, 0, 2], [3, 0, 2], [2, 1, 1], [0, 0, 2]],
}


def _tape(alloc, available):
    return {"available": available, "max": [[10], [4], [9]], "alloc": alloc}


SAMPLES = [
    {
        "name": "1) Safe state + request granted",
        "description": "Textbook example: 5 processes, 3 resources (A=10, B=5, C=7). P1 requests (1,0,2).",
        "state": _BASE,
        "request": {"pid": 1, "req": [1, 0, 2]},
        "expected": {"safe": True, "sequence": [1, 3, 4, 0, 2], "status": "granted", "code": "GRANTED"},
    },
    {
        "name": "2) Request denied: would be unsafe",
        "description": "After P1 got (1,0,2), P0 requests (0,2,0). Resources exist, but granting is unsafe.",
        "state": _AFTER_P1,
        "request": {"pid": 0, "req": [0, 2, 0]},
        "expected": {"safe": True, "sequence": [1, 3, 4, 0, 2], "status": "denied", "code": "UNSAFE"},
    },
    {
        "name": "3) Request exceeds Need",
        "description": "P1 has Need (1,2,2) but requests (3,0,0): more than it declared.",
        "state": _BASE,
        "request": {"pid": 1, "req": [3, 0, 0]},
        "expected": {"safe": True, "sequence": [1, 3, 4, 0, 2], "status": "denied", "code": "EXCEEDS_NEED"},
    },
    {
        "name": "4) Not enough available: wait",
        "description": "P4 requests (3,3,0) within its Need (4,3,1), but only (2,3,0) is free.",
        "state": _AFTER_P1,
        "request": {"pid": 4, "req": [3, 3, 0]},
        "expected": {"safe": True, "sequence": [1, 3, 4, 0, 2], "status": "wait", "code": "NOT_AVAILABLE"},
    },
    {
        "name": "5) One resource type: +1 makes it unsafe",
        "description": "12 tape drives (Max 10, 4, 9). Safe now, but giving P2 one more is unsafe.",
        "state": _tape([[5], [2], [2]], [3]),
        "request": {"pid": 2, "req": [1]},
        "expected": {"safe": True, "sequence": [1, 0, 2], "status": "denied", "code": "UNSAFE"},
    },
    {
        "name": "6) Starting state is already unsafe",
        "description": "Sample 5 after P2 got the extra drive anyway (no Banker). Only P1 can finish.",
        "state": _tape([[5], [2], [3]], [2]),
        "request": {"pid": 1, "req": [2]},
        "expected": {"safe": False, "sequence": [1], "status": "denied", "code": "UNSAFE"},
    },
]

In [ ]:
%%writefile script.py
"""Turns a scenario into a "video script": a list of Beats.

Each Beat has a narration caption, a chapter name, the View to show at the END
of the beat, and optional effects (flying numbers, staggered comparisons, ...).
The renderer animates from the previous beat's View to this one, so numbers
count up/down and colours fade automatically.
"""
import copy
from dataclasses import dataclass, field

import banker


def rname(j):
    return chr(65 + j) if j < 26 else f"R{j}"


def vec(v):
    return "(" + ", ".join(str(x) for x in v) + ")"


@dataclass
class View:
    n: int
    m: int
    alloc: list
    max: list
    need: list
    available: list
    rows_shown: float = 0          # how many table rows are visible (animates)
    need_shown: list = None        # per row: 0 hidden / 1 visible
    work: list = None              # None = Work box hidden
    request: list = None           # None = Request box hidden
    req_pid: int = None
    finish: list = None
    sequence: list = field(default_factory=list)
    row_hl: dict = field(default_factory=dict)   # row -> active|done|stuck|trial
    cell_hl: dict = field(default_factory=dict)  # cell key -> ok|bad|focus
    col_hl: set = field(default_factory=set)     # "alloc"/"max"/"need"/"avail"/"work"
    banner: tuple = None           # (text, kind) kind = ok|bad|wait|info
    title: tuple = None            # (big, small) -> full screen title card
    compare: tuple = None          # (left label, left vec, right label, right vec) shown under the boxes

    def copy(self, **kw):
        v = copy.deepcopy(self)
        for k, val in kw.items():
            setattr(v, k, val)
        return v


@dataclass
class Beat:
    chapter: str
    caption: str
    view: View
    effects: list = field(default_factory=list)
    duration: float = None   # seconds at 1x; None = from caption length

    def __post_init__(self):
        if self.duration is None:
            words = len(self.caption.split())
            self.duration = max(2.8, 0.9 + words * 0.32)


class _Builder:
    def __init__(self):
        self.beats = []
        self.v = None

    def beat(self, chapter, caption, effects=None, duration=None, **changes):
        self.v = self.v.copy(**changes)
        self.beats.append(Beat(chapter, caption, self.v, effects or [], duration))


def _safety_beats(b, state, chapter, label="Available"):
    """Append the beats that animate one run of the safety algorithm."""
    n, m = len(state["max"]), len(state["available"])
    res = banker.safety(state)
    need = banker.compute_need(state)
    last_pid = -1
    for st in res["steps"]:
        k = st["kind"]
        if k == "init":
            b.beat(chapter,
                   f"Start: copy Work = {label} = {vec(st['work'])}, and mark every process as not finished.",
                   effects=[("fly", [("avail", j) for j in range(m)], [("work", j) for j in range(m)])],
                   work=list(st["work"]), finish=[False] * n, sequence=[], row_hl={}, cell_hl={},
                   col_hl={"avail", "work"}, banner=None, compare=None)
            b.beat(chapter,
                   "Now we look for a process whose remaining Need fits inside Work. "
                   "If it fits, that process can run to completion.",
                   col_hl=set())
        elif k == "check":
            i = st["pid"]
            if i <= last_pid:
                b.beat(chapter, "End of the list. Some processes are still waiting, "
                                "so we go around again from the top with the bigger Work.",
                       row_hl={r: h for r, h in b.v.row_hl.items() if h == "done"}, cell_hl={}, compare=None,
                       duration=3.2)
            last_pid = i
            cells = {}
            for j in range(m):
                good = need[i][j] <= st["work"][j]
                cells[("need", i, j)] = "ok" if good else "bad"
                cells[("work", j)] = "ok" if good else "bad"
            hl = {r: h for r, h in b.v.row_hl.items() if h == "done"}
            hl[i] = "active"
            cmp = ("Need[P%d]" % i, need[i], "Work", st["work"])
            fails = [rname(j) for j in range(m) if need[i][j] > st["work"][j]]
            if st["ok"]:
                cap = (f"Check P{i}: Need {vec(need[i])} <= Work {vec(st['work'])}"
                       f"{' for every resource' if m > 1 else ''}. "
                       f"P{i} can finish!")
            else:
                cap = (f"Check P{i}: Need {vec(need[i])} vs Work {vec(st['work'])}. "
                       f"Not enough {', '.join(fails)}, so P{i} must wait. Skip it for now.")
            pairs = [(("need", i, j), ("work", j)) for j in range(m)]
            windows = {}
            for j, (a, c) in enumerate(pairs):
                w = (0.1 + 0.55 * j / m, 0.1 + 0.55 * (j + 1) / m)
                windows[a] = w
                windows[c] = w
            b.beat(chapter, cap, effects=[("stagger", windows)], row_hl=hl, cell_hl=cells, compare=cmp)
        elif k == "finish":
            i = st["pid"]
            hl = dict(b.v.row_hl)
            hl[i] = "done"
            b.beat(chapter,
                   f"P{i} runs, finishes, and gives back everything it holds: "
                   f"Work = {vec(st['before'])} + {vec(state['alloc'][i])} = {vec(st['work'])}. "
                   f"Add P{i} to the safe sequence.",
                   effects=[("fly", [("alloc", i, j) for j in range(m)], [("work", j) for j in range(m)])],
                   work=list(st["work"]), finish=list(st["finish"]), sequence=list(st["sequence"]),
                   row_hl=hl, cell_hl={}, compare=None, duration=None)
        elif k == "done":
            seq = ", ".join(f"P{p}" for p in st["sequence"])
            if st["safe"]:
                b.beat(chapter,
                       f"Every process can finish, so the state is SAFE. Safe sequence: <{seq}>.",
                       effects=[("pop",)], banner=("SAFE", "ok"), cell_hl={}, compare=None, duration=4.5)
            else:
                stuck = ", ".join(f"P{p}" for p in st["stuck"])
                hl = dict(b.v.row_hl)
                for p in st["stuck"]:
                    hl[p] = "stuck"
                b.beat(chapter,
                       f"Nobody in {{{stuck}}} has Need <= Work {vec(st['work'])}. "
                       "We can't guarantee they all finish: the state is UNSAFE (deadlock is possible).",
                       effects=[("pop",)], banner=("UNSAFE", "bad"), row_hl=hl, cell_hl={}, compare=None,
                       duration=5.5)
    return res


def build_script(sample):
    """Build the list of beats for one sample (or a custom state + optional request)."""
    state = sample["state"]
    n, m = len(state["max"]), len(state["available"])
    need = banker.compute_need(state)
    b = _Builder()
    b.v = View(n=n, m=m, alloc=copy.deepcopy(state["alloc"]), max=copy.deepcopy(state["max"]),
               need=copy.deepcopy(need), available=list(state["available"]), need_shown=[0] * n)
    res_names = ", ".join(rname(j) for j in range(m))

    # ---------------- Intro ----------------
    ch = "Introduction"
    b.beat(ch, "The Banker's Algorithm avoids deadlock: before giving resources to a process, "
               "the OS checks that everyone can still finish.",
           title=("Banker's Algorithm", sample.get("name", "Deadlock avoidance, step by step")), duration=5)
    b.beat(ch, sample.get("description") or f"{n} processes share {m} resource type(s).",
           title=("Banker's Algorithm", sample.get("name", "")), duration=5)
    b.beat(ch, f"Here are {n} processes, P0 to P{n - 1}, and {m} resource type{'s' if m > 1 else ''}: {res_names}.",
           title=None, rows_shown=n, effects=[("rows",)], duration=4)
    b.beat(ch, "Allocation: how many instances of each resource a process is holding right now.",
           col_hl={"alloc"})
    b.beat(ch, "Max: the most each process may ever ask for. Processes declare this in advance.",
           col_hl={"max"})
    b.beat(ch, f"Available: the free instances the OS still has. Right now that is {vec(state['available'])}.",
           col_hl={"avail"})

    # ---------------- Need ----------------
    ch = "Step 1: Compute Need"
    b.beat(ch, "First we compute Need = Max - Allocation: what each process may still request.",
           col_hl={"need"})
    shown = [0] * n
    detailed = min(n, 2)
    for i in range(detailed):
        shown = shown.copy()
        shown[i] = 1
        cells = {("max", i, j): "focus" for j in range(m)}
        cells.update({("alloc", i, j): "focus" for j in range(m)})
        b.beat(ch, f"P{i}: Max {vec(state['max'][i])} - Allocation {vec(state['alloc'][i])} = Need {vec(need[i])}.",
               need_shown=shown, cell_hl=cells, row_hl={i: "active"}, col_hl=set())
    if n > detailed:
        b.beat(ch, "The other processes are computed exactly the same way.",
               need_shown=[1] * n, cell_hl={}, row_hl={}, effects=[("stagger_rows", detailed)])
    b.beat(ch, "Now we have everything we need. Let's check if the system is in a safe state.",
           cell_hl={}, row_hl={}, col_hl={"need"}, duration=3.5)

    # ---------------- Safety ----------------
    res = _safety_beats(b, state, "Step 2: Safety Algorithm")

    # ---------------- Request ----------------
    rq = sample.get("request")
    if rq:
        ch = "Step 3: Resource Request"
        pid, req = rq["pid"], rq["req"]
        r = banker.request(state, pid, req)
        # reset the view to the real state
        b.beat(ch, f"Now a new event: P{pid} asks for {vec(req)}. Should the OS say yes?",
               request=list(req), req_pid=pid, work=None, finish=None, sequence=[], banner=None,
               row_hl={pid: "active"}, cell_hl={}, col_hl=set(), compare=None, duration=4)
        # check 1: request <= need
        ok1 = banker.leq(req, need[pid])
        cells = {}
        windows = {}
        for j in range(m):
            g = req[j] <= need[pid][j]
            cells[("req", j)] = cells[("need", pid, j)] = "ok" if g else "bad"
            windows[("req", j)] = windows[("need", pid, j)] = (0.1 + 0.55 * j / m, 0.1 + 0.55 * (j + 1) / m)
        b.beat(ch, f"Check 1: Request {vec(req)} <= Need {vec(need[pid])}? "
                   + ("Yes, P{} stays within what it declared.".format(pid) if ok1
                      else "No! P{} asks for more than it declared. That is an error.".format(pid)),
               cell_hl=cells, effects=[("stagger", windows)],
               compare=("Request", req, f"Need[P{pid}]", need[pid]))
        if not ok1:
            b.beat(ch, f"The request is DENIED immediately. P{pid} broke its promise about Max.",
                   effects=[("pop",)], banner=("DENIED", "bad"), cell_hl={}, compare=None, duration=4.5)
        else:
            ok2 = banker.leq(req, state["available"])
            cells, windows = {}, {}
            for j in range(m):
                g = req[j] <= state["available"][j]
                cells[("req", j)] = cells[("avail", j)] = "ok" if g else "bad"
                windows[("req", j)] = windows[("avail", j)] = (0.1 + 0.55 * j / m, 0.1 + 0.55 * (j + 1) / m)
            b.beat(ch, f"Check 2: Request {vec(req)} <= Available {vec(state['available'])}? "
                       + ("Yes, the resources exist right now." if ok2
                          else f"No. The OS doesn't have enough free right now, so P{pid} must WAIT."),
                   cell_hl=cells, effects=[("stagger", windows)],
                   compare=("Request", req, "Available", state["available"]))
            if not ok2:
                b.beat(ch, f"P{pid} WAITS until other processes release resources, then asks again.",
                       effects=[("pop",)], banner=("WAIT", "wait"), cell_hl={}, compare=None, duration=4.5)
            else:
                trial = r["trial"]
                t_need = banker.compute_need(trial)
                b.beat(ch, "Check 3: pretend we grant it. Available goes down, "
                           f"P{pid}'s Allocation goes up and its Need goes down by {vec(req)}.",
                       effects=[("fly", [("req", j) for j in range(m)], [("alloc", pid, j) for j in range(m)])],
                       available=list(trial["available"]), alloc=copy.deepcopy(trial["alloc"]),
                       need=t_need, cell_hl={("alloc", pid, j): "focus" for j in range(m)},
                       row_hl={pid: "trial"}, compare=None, duration=5)
                b.beat(ch, "This is only a trial state. Is it still safe? Run the Safety Algorithm again.",
                       cell_hl={}, duration=3.5)
                _safety_beats(b, trial, "Step 3: Safety check on trial state")
                if r["status"] == "granted":
                    b.beat("Result", f"The trial state is safe, so the request is GRANTED. "
                                     f"P{pid} gets {vec(req)} for real.",
                           effects=[("pop",)], banner=("GRANTED", "ok"), duration=4.5)
                else:
                    b.beat("Result", "The trial state is unsafe, so the request is DENIED. "
                                     f"Roll back: undo the pretend allocation, and P{pid} has to wait.",
                           effects=[("pop",)], banner=("DENIED", "bad"), duration=5)
                    b.beat("Result", "Rollback: Available, Allocation and Need return to their old values. "
                                     "Nothing was really given away.",
                           available=list(state["available"]), alloc=copy.deepcopy(state["alloc"]),
                           need=copy.deepcopy(need), work=None, finish=None, sequence=[], row_hl={},
                           banner=None, request=None, duration=5)

    # ---------------- Summary ----------------
    seq = ", ".join(f"P{p}" for p in res["sequence"])
    summary = (f"Summary: the starting state was {'SAFE, sequence <' + seq + '>' if res['safe'] else 'UNSAFE'}."
               + (f" Request from P{rq['pid']}: {banker.request(state, rq['pid'], rq['req'])['status'].upper()}."
                  if rq else ""))
    b.beat("Summary", summary, banner=None, row_hl={}, cell_hl={}, compare=None, duration=5)
    b.beat("Summary", "Remember: Banker grants a request only if the system can still find "
                      "an order in which every process finishes.",
           title=("That's the Banker's Algorithm!", "Need = Max - Allocation  |  Need <= Work  |  Work += Allocation"),
           duration=5)
    return b.beats

In [ ]:
%%writefile render.py
"""Draws one frame: the beat's View, animated from the previous View at progress t (0..1)."""
import math

import pygame

from script import rname

W, H = 1280, 720

BG = (15, 23, 42)
PANEL = (30, 41, 59)
CELL = (39, 52, 73)
LINE = (71, 85, 105)
TEXT = (226, 232, 240)
MUTED = (148, 163, 184)
GREEN = (34, 197, 94)
RED = (239, 68, 68)
AMBER = (245, 158, 11)
BLUE = (59, 130, 246)
PURPLE = (168, 85, 247)
GROUP = {"alloc": (56, 189, 248), "max": (244, 114, 182), "need": (250, 204, 21),
         "avail": (52, 211, 153), "work": (129, 140, 248), "req": (251, 146, 60)}
CELL_HL = {"ok": GREEN, "bad": RED, "focus": BLUE}
ROW_HL = {"active": (113, 90, 20), "done": (30, 35, 45), "stuck": (110, 30, 30), "trial": (80, 45, 120)}
BANNER = {"ok": GREEN, "bad": RED, "wait": AMBER, "info": BLUE}


def clamp(x, a=0.0, b=1.0):
    return a if x < a else b if x > b else x


def ease(x):
    x = clamp(x)
    return x * x * (3 - 2 * x)


def win(t, w):
    return ease((t - w[0]) / max(1e-6, w[1] - w[0]))


def mix(c1, c2, k):
    return tuple(int(a + (b - a) * k) for a, b in zip(c1, c2))


class Renderer:
    def __init__(self, surface):
        self.s = surface
        pygame.font.init()
        names = "dejavusans,segoeui,arial,helvetica,liberationsans"
        self.f = {sz: pygame.font.SysFont(names, sz) for sz in (16, 18, 22, 24, 28)}
        self.fb = {sz: pygame.font.SysFont(names, sz, bold=True) for sz in (16, 18, 20, 22, 26, 30, 44, 60, 84)}
        self.le = "≤" if self._has(self.fb[22], "≤") else "<="
        self.gt = ">"
        self.arrow = "→" if self._has(self.fb[22], "→") else "->"

    @staticmethod
    def _has(font, ch):
        m = font.metrics(ch)
        return bool(m) and m[0] is not None

    # ---------- small helpers ----------
    def text(self, txt, font, color, pos, anchor="center", alpha=255):
        img = font.render(str(txt), True, color)
        if alpha < 255:
            img.set_alpha(int(alpha))
        r = img.get_rect(**{anchor: pos})
        self.s.blit(img, r)
        return r

    def rect(self, color, r, radius=8, width=0, alpha=255):
        if alpha >= 255:
            pygame.draw.rect(self.s, color, r, width, border_radius=radius)
        else:
            tmp = pygame.Surface((r.w, r.h), pygame.SRCALPHA)
            pygame.draw.rect(tmp, (*color, int(alpha)), tmp.get_rect(), width, border_radius=radius)
            self.s.blit(tmp, r.topleft)

    def wrap(self, txt, font, width):
        lines, cur = [], ""
        for word in txt.split(" "):
            test = (cur + " " + word).strip()
            if font.size(test)[0] <= width:
                cur = test
            else:
                lines.append(cur)
                cur = word
        lines.append(cur)
        return lines

    # ---------- layout ----------
    def layout(self, v):
        n, m = v.n, v.m
        L = {}
        x0, y0 = 40, 92
        name_w, fin_w, gap = 64, 76, 16
        cw = int(min(62, (830 - name_w - fin_w - 3 * gap) / (3 * m)))
        cw = max(cw, -(-110 // m))  # keep group titles from overlapping when m is small
        rh = int(min(54, (560 - y0 - 58) / n))
        L["cw"], L["rh"] = cw, rh
        L["head_y"] = y0
        gx = x0 + name_w
        L["groups"] = {}
        for g in ("alloc", "max", "need"):
            L["groups"][g] = pygame.Rect(gx, y0, cw * m, 58 + rh * n)
            for i in range(n):
                for j in range(m):
                    L[(g, i, j)] = pygame.Rect(gx + j * cw + 2, y0 + 58 + i * rh + 2, cw - 4, rh - 4)
            gx += cw * m + gap
        L["finish_x"] = gx
        L["fin_w"] = fin_w
        L["table"] = pygame.Rect(x0, y0, gx + fin_w - x0, 58 + rh * n)
        for i in range(n):
            L[("row", i)] = pygame.Rect(x0 - 6, y0 + 58 + i * rh, gx + fin_w - x0 + 12, rh)
            L[("name", i)] = (x0 + name_w // 2, y0 + 58 + i * rh + rh // 2)
        # right panel
        px = 905
        bw = int(min(58, 320 / m - 6))
        for k, y in (("avail", 120), ("work", 220), ("req", 320)):
            for j in range(m):
                L[(k, j)] = pygame.Rect(px + j * (bw + 6), y, bw, 46)
            L[(k, "label")] = (px, y - 24)
        L["cmp"] = pygame.Rect(px - 10, 395, 345, 100)
        L["seq"] = pygame.Rect(px - 10, 505, 345, 80)
        return L

    def cell_center(self, L, key):
        return L[key].center

    # ---------- frame ----------
    def draw(self, beat, prev, t, info):
        """info: dict(chapter, progress, chapters[(frac, name)], caption_k, paused, speed, sample)"""
        v = beat.view
        p = prev if prev is not None else v
        fx = {e[0]: e for e in beat.effects}
        self.s.fill(BG)

        # title card replaces the whole stage
        if v.title is not None:
            a = 1.0 if p.title is not None else win(t, (0, 0.3))
            self.draw_title(v.title, a)
            self.draw_caption(beat.caption, info)
            self.draw_progress(info)
            return

        L = self.layout(v)
        num_win = {}
        if "fly" in fx:
            for k in fx["fly"][2]:
                num_win[k] = (0.45, 0.85)
        stag = fx["stagger"][1] if "stagger" in fx else {}

        def num(key, cur, old):
            if old is None or old == cur:
                return cur
            return round(old + (cur - old) * win(t, num_win.get(key, (0.15, 0.8))))

        def hl_color(key, base):
            w = stag.get(key, (0.0, 0.3))
            c_new = CELL_HL.get(v.cell_hl.get(key), base)
            c_old = CELL_HL.get(p.cell_hl.get(key), base)
            return mix(c_old, c_new, win(t, w))

        self.draw_header(info)
        rows_shown = p.rows_shown + (v.rows_shown - p.rows_shown) * win(t, (0, 0.9))

        # column-group highlight
        for g, r in L["groups"].items():
            k = (1 if g in v.col_hl else 0) * win(t, (0, 0.4)) + (1 if g in p.col_hl else 0) * (1 - win(t, (0, 0.4)))
            if k > 0.01:
                self.rect(GROUP[g], r.inflate(10, 10), 10, alpha=40 * k)
                self.rect(GROUP[g], r.inflate(10, 10), 10, width=2, alpha=255 * k)

        # headers
        hy = L["head_y"]
        self.text("Process", self.fb[16], MUTED, (L[("name", 0)][0], hy + 40))
        for g, label in (("alloc", "Allocation"), ("max", "Max"), ("need", "Need")):
            r = L["groups"][g]
            self.text(label, self.fb[20], GROUP[g], (r.centerx, hy + 14))
            for j in range(v.m):
                self.text(rname(j), self.fb[16], MUTED, (r.x + j * L["cw"] + L["cw"] // 2, hy + 42))
        self.text("Finish", self.fb[16], MUTED, (L["finish_x"] + L["fin_w"] // 2, hy + 40))

        # rows
        for i in range(v.n):
            a = clamp(rows_shown - i)
            if a <= 0:
                continue
            off = int((1 - ease(a)) * 40)
            rr = L[("row", i)].move(off, 0)
            c_old = ROW_HL.get(p.row_hl.get(i))
            c_new = ROW_HL.get(v.row_hl.get(i))
            k = win(t, (0, 0.3))
            if c_old or c_new:
                c = mix(c_old or BG, c_new or BG, k)
                self.rect(c, rr, 10, alpha=255 * a)
            done = v.row_hl.get(i) == "done"
            name_col = MUTED if done else TEXT
            self.text(f"P{i}", self.fb[22], name_col, (L[("name", i)][0] + off, L[("name", i)][1]), alpha=255 * a)
            for g in ("alloc", "max", "need"):
                vals_new = getattr(v, g)[i]
                vals_old = getattr(p, g)[i]
                if g == "need":
                    na = (p.need_shown[i] + (v.need_shown[i] - p.need_shown[i]) *
                          win(t, self._need_window(fx, i))) * a
                else:
                    na = a
                for j in range(v.m):
                    key = (g, i, j)
                    r = L[key].move(off, 0)
                    self.rect(hl_color(key, CELL), r, 6, alpha=255 * a)
                    if na > 0:
                        self.text(num(key, vals_new[j], vals_old[j]), self.fb[22],
                                  MUTED if done else TEXT, r.center, alpha=255 * na)
            # finish column
            fc = (L["finish_x"] + L["fin_w"] // 2 + off, L[("name", i)][1])
            if v.finish is not None:
                fin = v.finish[i]
                was = p.finish[i] if p.finish is not None else False
                if fin:
                    k2 = 1 if was else win(t, (0.6, 0.9))
                    self.text("true", self.fb[18], mix(MUTED, GREEN, k2), fc, alpha=255 * a)
                else:
                    self.text("false", self.f[18], RED if v.row_hl.get(i) == "stuck" else MUTED, fc, alpha=255 * a)

        # right panel boxes
        self.draw_vector(L, "avail", "Available", v.available, p.available, num, hl_color, 1.0)
        if v.work is not None or p.work is not None:
            if v.work is not None:
                a = 1.0 if p.work is not None else win(t, (0, 0.3))
                self.draw_vector(L, "work", "Work", v.work, p.work if p.work is not None else [0] * v.m,
                                 num, hl_color, a)
            else:
                self.draw_vector(L, "work", "Work", p.work, p.work, num, hl_color, 1 - win(t, (0, 0.3)))
        if v.request is not None:
            a = 1.0 if p.request is not None else win(t, (0, 0.3))
            self.draw_vector(L, "req", f"Request from P{v.req_pid}", v.request, v.request, num, hl_color, a)
        elif p.request is not None:
            self.draw_vector(L, "req", f"Request from P{p.req_pid}", p.request, p.request, num, hl_color,
                             1 - win(t, (0, 0.3)))

        if v.compare is not None:
            self.draw_compare(L, v.compare, stag, t, 1.0 if p.compare == v.compare else win(t, (0, 0.2)))
        self.draw_sequence(L, v, p, t)

        if "fly" in fx:
            self.draw_fly(L, fx["fly"], p, t)

        if v.banner is not None:
            popped = "pop" in fx and p.banner != v.banner
            self.draw_banner(L, v.banner, win(t, (0, 0.25)) if popped else 1.0, t if popped else 1.0)
        elif p.banner is not None:
            self.draw_banner(L, p.banner, 1 - win(t, (0, 0.2)), 1.0)

        self.draw_caption(beat.caption, info)
        self.draw_progress(info)

    def _need_window(self, fx, i):
        if "stagger_rows" in fx:
            d = fx["stagger_rows"][1]
            return (0.1 + 0.08 * (i - d), 0.3 + 0.08 * (i - d))
        return (0.2, 0.6)

    # ---------- parts ----------
    def draw_header(self, info):
        self.rect(PANEL, pygame.Rect(0, 0, W, 60), 0)
        self.text(info["chapter"], self.fb[26], TEXT, (32, 30), "midleft")
        self.text(info["sample"], self.f[18], MUTED, (W - 32, 30), "midright")

    def draw_vector(self, L, k, label, vals, old, num, hl_color, alpha):
        if alpha <= 0.01:
            return
        self.text(label, self.fb[20], GROUP[k], L[(k, "label")], "topleft", alpha=255 * alpha)
        for j, val in enumerate(vals):
            r = L[(k, j)]
            self.rect(hl_color((k, j), CELL), r, 8, alpha=255 * alpha)
            self.rect(GROUP[k], r, 8, width=2, alpha=255 * alpha)
            self.text(num((k, j), val, old[j] if old is not None else None), self.fb[26], TEXT, r.center,
                      alpha=255 * alpha)
            self.text(rname(j), self.f[16], MUTED, (r.centerx, r.bottom + 10), alpha=200 * alpha)

    def draw_compare(self, L, cmp, stag, t, alpha):
        left_l, left, right_l, right = cmp
        box = L["cmp"]
        self.rect(PANEL, box, 12, alpha=255 * alpha)
        self.text("Compare each resource", self.f[16], MUTED, (box.x + 12, box.y + 8), "topleft", alpha=255 * alpha)
        m = len(left)
        colw = min(48, (box.w - 120) // m)
        x0 = box.x + 112
        self.text(left_l, self.fb[16], TEXT, (box.x + 12, box.y + 42), "midleft", alpha=255 * alpha)
        self.text(right_l, self.fb[16], TEXT, (box.x + 12, box.y + 82), "midleft", alpha=255 * alpha)
        windows = list(stag.values())
        for j in range(m):
            cx = x0 + j * colw + colw // 2
            w = (0.1 + 0.55 * j / m, 0.1 + 0.55 * (j + 1) / m)
            k = win(t, w) if windows else 1
            ok = left[j] <= right[j]
            col = mix(TEXT, GREEN if ok else RED, k)
            self.text(left[j], self.fb[22], col, (cx, box.y + 42), alpha=255 * alpha)
            self.text(right[j], self.fb[22], col, (cx, box.y + 82), alpha=255 * alpha)
            if k > 0:
                self.text(self.le if ok else self.gt, self.fb[20], col, (cx, box.y + 62), alpha=255 * alpha * k)

    def draw_sequence(self, L, v, p, t):
        box = L["seq"]
        if v.finish is None:
            return
        self.text("Safe sequence", self.fb[20], GREEN, (box.x + 10, box.y - 6), "topleft")
        x, y = box.x + 10, box.y + 24
        for idx, pid in enumerate(v.sequence):
            new = idx >= len(p.sequence)
            a = win(t, (0.55, 0.9)) if new else 1
            chip = pygame.Rect(x, y, 50, 32)
            if chip.right > box.right:
                x, y = box.x + 10, y + 40
                chip = pygame.Rect(x, y, 50, 32)
            chip = chip.move(0, int((1 - a) * -20))
            self.rect(GREEN, chip, 16, alpha=200 * a)
            self.text(f"P{pid}", self.fb[18], BG, chip.center, alpha=255 * a)
            x += 58

    def draw_fly(self, L, fx, p, t):
        _, srcs, dsts = fx
        k = win(t, (0.05, 0.55))
        if k <= 0 or k >= 1:
            return
        for s, d in zip(srcs, dsts):
            val = self._value(p, s)
            a, b = pygame.Vector2(L[s].center), pygame.Vector2(L[d].center)
            pos = a.lerp(b, k)
            pos.y -= math.sin(k * math.pi) * 60  # arc
            col = GROUP.get(s[0], AMBER)
            r = pygame.Rect(0, 0, 40, 34)
            r.center = (int(pos.x), int(pos.y))
            self.rect(col, r, 17)
            self.text(f"+{val}" if d[0] in ("work", "alloc") else val, self.fb[20], BG, r.center)

    @staticmethod
    def _value(v, key):
        if key[0] in ("alloc", "max", "need"):
            return getattr(v, key[0])[key[1]][key[2]]
        return {"avail": v.available, "work": v.work, "req": v.request}[key[0]][key[1]]

    def draw_banner(self, L, banner, a, t):
        if a <= 0.01:
            return
        txt, kind = banner
        col = BANNER[kind]
        tbl = L["table"]
        scale = 0.4 + 0.6 * a + (0.12 * math.sin(clamp(t / 0.25) * math.pi) if t < 0.25 else 0)
        img = self.fb[84].render(txt, True, (255, 255, 255))
        img = pygame.transform.smoothscale_by(img, scale) if hasattr(pygame.transform, "smoothscale_by") \
            else pygame.transform.smoothscale(img, (int(img.get_width() * scale), int(img.get_height() * scale)))
        # below the table when there is room, so the numbers stay readable
        cy = (tbl.bottom + 565) // 2 if tbl.bottom <= 430 else tbl.centery
        r = img.get_rect(center=(tbl.centerx, cy))
        self.rect(col, r.inflate(70, 30), 20, alpha=230 * a)
        img.set_alpha(int(255 * a))
        self.s.blit(img, r)

    def draw_title(self, title, a):
        big, small = title
        for y in range(0, 580, 4):
            c = mix((30, 27, 75), BG, y / 580)
            pygame.draw.rect(self.s, c, (0, y, W, 4))
        self.text(big, self.fb[60], TEXT, (W // 2, 250 - int((1 - a) * 30)), alpha=255 * a)
        if small:
            self.text(small, self.f[28], (165, 180, 252), (W // 2, 330), alpha=255 * a)
        pygame.draw.line(self.s, GROUP["work"], (W // 2 - 140 * a, 290), (W // 2 + 140 * a, 290), 3)

    def draw_caption(self, caption, info):
        box = pygame.Rect(30, 596, W - 60, 92)
        self.rect((2, 6, 23), box, 12, alpha=235)
        shown = caption[: int(len(caption) * info["caption_k"])]
        lines = self.wrap(caption, self.f[24], box.w - 40)
        used = 0
        y = box.y + 14 + (3 - len(lines)) * 13 if len(lines) < 3 else box.y + 8
        for line in lines:
            part = shown[used: used + len(line)]
            used += len(line) + 1
            if part:
                self.text(part, self.f[24], TEXT, (box.x + 20, y), "topleft")
            y += 28

    def draw_progress(self, info):
        bar = pygame.Rect(30, 700, W - 60, 8)
        self.rect(LINE, bar, 4)
        self.rect(RED, pygame.Rect(bar.x, bar.y, int(bar.w * info["progress"]), bar.h), 4)
        for frac, _ in info["chapters"]:
            x = bar.x + int(bar.w * frac)
            pygame.draw.rect(self.s, BG, (x - 1, bar.y, 3, bar.h))
        state = "PAUSED" if info["paused"] else f"{info['speed']:g}x"
        if info.get("hud", True):
            self.text(f"{state}   Space pause   <- -> step   Up/Down speed   1-{info['nsamples']} sample   R restart",
                      self.f[16], MUTED, (W - 32, 572), "midright")

In [ ]:
%%writefile main.py
"""Banker's Algorithm — animated explainer (pygame).

    python main.py                     play sample 1 in a window
    python main.py --sample 2          pick a sample (1-6)
    python main.py --json state.json   your own data (same JSON as the web version's "save")
    python main.py --record out.mp4    render a video file instead of opening a window (needs ffmpeg)

Keys: Space pause/play, Left/Right previous/next step, Up/Down speed,
      1-6 switch sample, R restart, H hide/show key hints, Esc quit.
      Click the progress bar to jump.
"""
import argparse
import json
import os
import shutil
import subprocess
import sys

import banker
from samples import SAMPLES
from script import build_script

FPS = 60


class Player:
    def __init__(self, sample):
        self.load(sample)
        self.speed = 1.0
        self.paused = False
        self.hud = True

    def load(self, sample):
        self.sample = sample
        self.beats = build_script(sample)
        self.starts = []
        acc = 0.0
        for b in self.beats:
            self.starts.append(acc)
            acc += b.duration
        self.total = acc
        self.idx = 0
        self.t = 0.0  # seconds into current beat
        self.chapters = []
        last = None
        for b, s in zip(self.beats, self.starts):
            if b.chapter != last:
                self.chapters.append((s / self.total, b.chapter))
                last = b.chapter

    @property
    def finished(self):
        return self.idx == len(self.beats) - 1 and self.t >= self.beats[-1].duration

    def update(self, dt):
        if self.paused or self.finished:
            return
        self.t += dt * self.speed
        while self.t >= self.beats[self.idx].duration and self.idx < len(self.beats) - 1:
            self.t -= self.beats[self.idx].duration
            self.idx += 1
        self.t = min(self.t, self.beats[self.idx].duration)

    def jump(self, idx, at_end=False):
        self.idx = max(0, min(len(self.beats) - 1, idx))
        self.t = self.beats[self.idx].duration if at_end else 0.0

    def seek(self, frac):
        target = frac * self.total
        for i, s in enumerate(self.starts):
            if s <= target < s + self.beats[i].duration:
                self.idx, self.t = i, target - s
                return

    def frame_info(self):
        b = self.beats[self.idx]
        k = min(1.0, self.t / min(1.6, b.duration * 0.45))
        return {
            "chapter": b.chapter,
            "progress": (self.starts[self.idx] + self.t) / self.total,
            "chapters": self.chapters,
            "caption_k": k,
            "paused": self.paused,
            "speed": self.speed,
            "sample": self.sample.get("name", ""),
            "nsamples": len(SAMPLES),
            "hud": self.hud,
        }

    def draw(self, renderer):
        b = self.beats[self.idx]
        prev = self.beats[self.idx - 1].view if self.idx > 0 else None
        renderer.draw(b, prev, self.t / b.duration, self.frame_info())


def load_json(path):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    state = {"available": data["available"], "max": data["max"], "alloc": data["alloc"]}
    errors = banker.validate_state(state)
    if errors:
        sys.exit("Invalid input:\n  " + "\n  ".join(errors))
    sample = {"name": os.path.basename(path), "description": "", "state": state}
    if "request" in data:
        sample["request"] = {"pid": data["request"]["pid"], "req": data["request"]["req"]}
    return sample


def run_window(player):
    import pygame
    from render import W, H, Renderer

    pygame.init()
    screen = pygame.display.set_mode((W, H), pygame.SCALED | pygame.RESIZABLE)
    pygame.display.set_caption("Banker's Algorithm — animated")
    renderer = Renderer(screen)
    clock = pygame.time.Clock()
    speeds = [0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 3.0]
    while True:
        dt = clock.tick(FPS) / 1000
        for e in pygame.event.get():
            if e.type == pygame.QUIT:
                return
            if e.type == pygame.KEYDOWN:
                if e.key in (pygame.K_ESCAPE, pygame.K_q):
                    return
                elif e.key == pygame.K_SPACE:
                    if player.finished:
                        player.jump(0)
                        player.paused = False
                    else:
                        player.paused = not player.paused
                elif e.key == pygame.K_RIGHT:
                    # finish the current beat first, then move on
                    if player.t < player.beats[player.idx].duration - 1e-3:
                        player.jump(player.idx, at_end=True)
                    else:
                        player.jump(player.idx + 1, at_end=player.paused)
                elif e.key == pygame.K_LEFT:
                    player.jump(player.idx - (1 if player.t < 0.6 else 0), at_end=player.paused)
                elif e.key == pygame.K_UP:
                    player.speed = next((s for s in speeds if s > player.speed), speeds[-1])
                elif e.key == pygame.K_DOWN:
                    player.speed = next((s for s in reversed(speeds) if s < player.speed), speeds[0])
                elif e.key == pygame.K_r:
                    player.jump(0)
                    player.paused = False
                elif e.key == pygame.K_h:
                    player.hud = not player.hud
                elif pygame.K_1 <= e.key <= pygame.K_9 and e.key - pygame.K_1 < len(SAMPLES):
                    player.load(SAMPLES[e.key - pygame.K_1])
                    player.paused = False
            if e.type == pygame.MOUSEBUTTONDOWN and e.button == 1 and 690 <= e.pos[1] <= 718:
                player.seek(max(0.0, min(0.999, (e.pos[0] - 30) / (W - 60))))
        player.update(dt)
        player.draw(renderer)
        pygame.display.flip()


def record(players, out, fps=30):
    if not shutil.which("ffmpeg"):
        sys.exit("ffmpeg not found. Install it (https://ffmpeg.org) to export video.")
    os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
    import pygame
    from render import W, H, Renderer

    pygame.init()
    pygame.display.set_mode((1, 1))
    surface = pygame.Surface((W, H))
    renderer = Renderer(surface)
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{W}x{H}",
           "-r", str(fps), "-i", "-", "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "20", out]
    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE)
    tobytes = getattr(pygame.image, "tobytes", None) or pygame.image.tostring
    frames = 0
    for player in players:
        player.hud = False
        while True:
            player.draw(renderer)
            proc.stdin.write(tobytes(surface, "RGB"))
            frames += 1
            if player.finished:
                break
            player.update(1 / fps)
        for _ in range(fps):  # hold the last frame for a second
            proc.stdin.write(tobytes(surface, "RGB"))
            frames += 1
        print(f"  rendered {player.sample['name']}")
    proc.stdin.close()
    proc.wait()
    print(f"Saved {out} ({frames / fps:.0f} s)")


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--sample", type=int, default=1, help=f"sample number 1-{len(SAMPLES)}")
    ap.add_argument("--json", help="load state (+ optional request) from a JSON file")
    ap.add_argument("--record", metavar="OUT.mp4", help="render to a video file instead of a window")
    ap.add_argument("--all", action="store_true", help="with --record: put every sample in one video")
    ap.add_argument("--speed", type=float, default=1.0, help="playback speed (default 1.0)")
    args = ap.parse_args()

    if args.json:
        samples = [load_json(args.json)]
    elif args.all:
        samples = SAMPLES
    else:
        if not 1 <= args.sample <= len(SAMPLES):
            ap.error(f"--sample must be 1-{len(SAMPLES)}")
        samples = [SAMPLES[args.sample - 1]]

    players = [Player(s) for s in samples]
    for p in players:
        p.speed = args.speed
    if args.record:
        record(players, args.record)
    else:
        run_window(players[0])


if __name__ == "__main__":
    main()

## 2. Choose the data

In [ ]:
#@title Pick a sample, or use your own data
#@markdown **Samples:** 1) safe + granted · 2) denied (unsafe) · 3) exceeds Need · 4) wait · 5) one resource, unsafe · 6) already unsafe
sample_number = 1  #@param {type:"slider", min:1, max:6, step:1}
use_my_own_data = False  #@param {type:"boolean"}
#@markdown ---
#@markdown **Your own data** (used only when the box above is ticked). Numbers separated by spaces, rows by `;`
available = "3 3 2"  #@param {type:"string"}
max_matrix = "7 5 3; 3 2 2; 9 0 2; 2 2 2; 4 3 3"  #@param {type:"string"}
allocation = "0 1 0; 2 0 0; 3 0 2; 2 1 1; 0 0 2"  #@param {type:"string"}
#@markdown Request (leave `request` empty to skip the request part)
request_process = 1  #@param {type:"integer"}
request = "1 0 2"  #@param {type:"string"}

import importlib, banker, samples
importlib.reload(banker); importlib.reload(samples)

def _vec(s):
    return [int(x) for x in s.replace(",", " ").split()]

def _mat(s):
    return [_vec(r) for r in s.split(";") if r.strip()]

if use_my_own_data:
    state = {"available": _vec(available), "max": _mat(max_matrix), "alloc": _mat(allocation)}
    errors = banker.validate_state(state)
    if errors:
        raise ValueError("Please fix the data:\n  " + "\n  ".join(errors))
    chosen = {"name": "My data", "description": "", "state": state}
    if request.strip():
        chosen["request"] = {"pid": request_process, "req": _vec(request)}
else:
    chosen = samples.SAMPLES[sample_number - 1]

need = banker.compute_need(chosen["state"])
res = banker.safety(chosen["state"])
print("Scenario :", chosen["name"])
print("Need     :", need)
print("Safe?    :", res["safe"], "  sequence:", ["P%d" % p for p in res["sequence"]])
if "request" in chosen:
    r = banker.request(chosen["state"], chosen["request"]["pid"], chosen["request"]["req"])
    print("Request  : P%d asks %s -> %s (%s)" % (chosen["request"]["pid"], chosen["request"]["req"],
                                               r["status"].upper(), r["code"]))

## 3. Watch the video

In [ ]:
#@title Render the video and play it (takes about a minute)
speed = 1.0  #@param {type:"slider", min:0.5, max:3, step:0.25}
fps = 24  #@param [24, 30] {type:"raw"}
download_after = False  #@param {type:"boolean"}

import importlib, base64, main, script, render
for mod in (script, render, main):
    importlib.reload(mod)
from IPython.display import HTML, display

player = main.Player(chosen)
player.speed = speed
print(f"{len(player.beats)} steps, about {player.total / speed / 60:.1f} min of video. Rendering...")
main.record([player], "banker.mp4", fps=fps)

with open("banker.mp4", "rb") as f:
    data = base64.b64encode(f.read()).decode()
display(HTML(f'<video width="960" controls autoplay><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))

if download_after:
    from google.colab import files
    files.download("banker.mp4")

## 4. Step through (no video needed)

In [ ]:
#@title Step-by-step viewer: press ▶ or drag the slider
import importlib, io, main, script, render
for mod in (script, render, main):
    importlib.reload(mod)
import pygame
import ipywidgets as widgets
from IPython.display import display

pygame.init()
pygame.display.set_mode((1, 1))
_surface = pygame.Surface((render.W, render.H))
_renderer = render.Renderer(_surface)
_viewer = main.Player(chosen)
_viewer.hud = False

def _frame(i):
    _viewer.idx = i
    _viewer.t = _viewer.beats[i].duration   # end of the step: animation done, caption complete
    _viewer.draw(_renderer)
    buf = io.BytesIO()
    pygame.image.save(_surface, buf, "frame.png")
    return buf.getvalue()

_frames = [_frame(i) for i in range(len(_viewer.beats))]
n = len(_frames)
image = widgets.Image(value=_frames[0], format="png", width=960)
slider = widgets.IntSlider(min=0, max=n - 1, description="Step", layout=widgets.Layout(width="600px"))
play = widgets.Play(min=0, max=n - 1, interval=3500, description="Play")
widgets.jslink((play, "value"), (slider, "value"))
label = widgets.HTML()

def _show(change=None):
    i = slider.value
    image.value = _frames[i]
    b = _viewer.beats[i]
    label.value = f"<b>{i + 1}/{n} · {b.chapter}</b><br>{b.caption}"

slider.observe(_show, names="value")
_show()
display(widgets.VBox([widgets.HBox([play, slider]), label, image]))

---
**Tip:** for a single video with all 6 samples, run:
```python
main.record([main.Player(s) for s in samples.SAMPLES], "banker_all.mp4", fps=24)
```